# Medical Deepfake Detection with ViT-B/16

This notebook uses the processed dataset already uploaded to Google Drive at `capstone_v1/processed/`. It expects `train/` and `val/` folders containing `fake/` and `real/` image subfolders.

A binary detector needs both genuine and synthetic examples; synthetic scans alone are not enough to train it.


In [1]:
# from google.colab import drive
from pathlib import Path

# drive.mount('/content/drive')

PROJECT_DIR = Path(r'C:\Users\aayus\OneDrive\Desktop\capstonevit\capstone_v1\medical_deepfake_data')

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Cannot find the project folder: {PROJECT_DIR}')

# Locate the already-uploaded processed folder anywhere inside capstone_v1.
processed_dirs = [path for path in PROJECT_DIR.rglob('processed') if path.is_dir()]
if not processed_dirs:
    raise FileNotFoundError(f'No processed folder was found inside {PROJECT_DIR}.')
if len(processed_dirs) > 1:
    print('Multiple processed folders found; using:', processed_dirs[0])

PROCESSED_DIR = processed_dirs[0]

In [2]:
# Run once in a new environment
%pip install -q transformers datasets torch torchvision scikit-learn pillow matplotlib nibabel accelerate

from pathlib import Path
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoImageProcessor, ViTForImageClassification, TrainingArguments, Trainer



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\aayus\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# The processed data is already on Drive, so no preparation script is needed.
# `PROCESSED_DIR` was set in the Drive-mount cell above.

print(f'Using processed dataset: {PROCESSED_DIR}')


Using processed dataset: C:\Users\aayus\OneDrive\Desktop\capstonevit\capstone_v1\medical_deepfake_data\processed


In [ ]:
from pathlib import Path
from collections import Counter

from datasets import Dataset, Features, ClassLabel, Image as HFImage


# ============================================================
# DATASET
# ============================================================

CLASS_NAMES = ["fake", "real"]

TRAIN_DIR = PROCESSED_DIR / "train"
VAL_DIR = PROCESSED_DIR / "val"


def collect_split(split_dir: Path):
    records = []

    for label_name in CLASS_NAMES:

        label_dir = split_dir / label_name

        if not label_dir.is_dir():
            raise FileNotFoundError(
                f"Missing directory: {label_dir}"
            )

        for image_path in sorted(label_dir.glob("*.png")):

            records.append(
                (
                    str(image_path),
                    label_name,
                )
            )

    return records


train_records = collect_split(TRAIN_DIR)
val_records = collect_split(VAL_DIR)


if not train_records:
    raise RuntimeError("Training dataset is empty.")

if not val_records:
    raise RuntimeError("Validation dataset is empty.")


label2id = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

id2label = {
    index: name
    for name, index in label2id.items()
}


train_paths = [
    record[0]
    for record in train_records
]

train_labels = [
    label2id[record[1]]
    for record in train_records
]

val_paths = [
    record[0]
    for record in val_records
]

val_labels = [
    label2id[record[1]]
    for record in val_records
]


print("=" * 60)
print("DATASET")
print("=" * 60)

print("Training images:", len(train_records))
print("Validation images:", len(val_records))

print(
    "Training classes:",
    dict(Counter(train_labels))
)

print(
    "Validation classes:",
    dict(Counter(val_labels))
)


# ============================================================
# DATASETS
# ============================================================

features = Features({
    "image": HFImage(),
    "label": ClassLabel(names=CLASS_NAMES),
})


train_ds = Dataset.from_dict(
    {
        "image": train_paths,
        "label": train_labels,
    },
    features=features,
)


val_ds = Dataset.from_dict(
    {
        "image": val_paths,
        "label": val_labels,
    },
    features=features,
)


print()
print("Train dataset:", train_ds)
print("Validation dataset:", val_ds)

Total images found: 12190
Class counts (all data): {'fake': 1996, 'real': 10194}
Source scans — train: 1951 | validation: 345
Train class counts: {0: 1696, 1: 8665}
Val class counts: {0: 300, 1: 1529}
Train: 10361 | Validation: 1829 | Labels: ['fake', 'real']


## Diagnostic: check for trivial (non-anatomical) shortcuts

The split above already rules out slice-level leakage between train and val. If accuracy is still ~1.0 immediately, the next most likely cause is a **shortcut**: some low-level property — resolution, file format, compression noise, a border/margin, a GAN checkerboard pattern — that differs between `real` and `fake` and has nothing to do with anatomy. This cell summarizes exactly that, per class and per split, so you can catch it before spending a training run on it.


In [5]:
from PIL import Image as PILImage
from collections import Counter as _Counter

def image_stats(paths, max_n=300):
    sizes, modes, means, stds = [], [], [], []
    for p in paths[:max_n]:
        with PILImage.open(p) as img:
            sizes.append(img.size)
            modes.append(img.mode)
            arr = np.array(img.convert('L'), dtype=np.float32)
            means.append(arr.mean())
            stds.append(arr.std())
    return {
        'n_checked': len(sizes),
        'sizes': _Counter(sizes).most_common(5),
        'modes': _Counter(modes).most_common(5),
        'mean_intensity (avg, std-across-images)': (float(np.mean(means)), float(np.std(means))),
        'per_image_std_intensity (avg, std-across-images)': (float(np.mean(stds)), float(np.std(stds))),
    }

for split_name, paths_list, labels_list in [
    ('train', train_paths, train_labels),
    ('val', val_paths, val_labels),
]:
    print(f'--- {split_name} ---')
    for label_id, label_name in enumerate(CLASS_NAMES):
        class_paths = [p for p, l in zip(paths_list, labels_list) if l == label_id]
        stats = image_stats(class_paths)
        print(f'{label_name}: {stats}')
    print()

# Red flags to look for in the printed output:
#  - 'sizes' differs consistently between real and fake (e.g. fake is always
#    256x256 while real is always 240x240) -> the model can trivially use
#    resolution as the label instead of content.
#  - 'modes' differs (e.g. fake images are saved as 'L' and real as 'RGB'
#    before the .convert('RGB') call) -> same problem.
#  - mean/std intensity ranges are cleanly separated between classes with
#    little overlap -> also a strong non-anatomical shortcut.
# If you see a clean shortcut here, fix it upstream in prepare_dataset.py
# (resize/normalize both classes identically, strip format-specific
# metadata, match compression) and re-run dataset preparation rather than
# patching it in this notebook.


--- train ---
fake: {'n_checked': 300, 'sizes': [((256, 256), 300)], 'modes': [('RGB', 300)], 'mean_intensity (avg, std-across-images)': (89.88882446289062, 29.831768035888672), 'per_image_std_intensity (avg, std-across-images)': (80.1521224975586, 14.716510772705078)}
real: {'n_checked': 300, 'sizes': [((256, 256), 300)], 'modes': [('RGB', 300)], 'mean_intensity (avg, std-across-images)': (93.2242660522461, 32.8203125), 'per_image_std_intensity (avg, std-across-images)': (75.9741439819336, 21.997896194458008)}

--- val ---
fake: {'n_checked': 300, 'sizes': [((256, 256), 300)], 'modes': [('RGB', 300)], 'mean_intensity (avg, std-across-images)': (86.38583374023438, 29.02216339111328), 'per_image_std_intensity (avg, std-across-images)': (78.9285888671875, 14.998607635498047)}
real: {'n_checked': 300, 'sizes': [((256, 256), 300)], 'modes': [('RGB', 300)], 'mean_intensity (avg, std-across-images)': (95.64318084716797, 31.860048294067383), 'per_image_std_intensity (avg, std-across-images)':

In [6]:
# --- Diagnostic 1b: background/framing fraction check ---
# Percentile normalization equalizes contrast range, not overall brightness.
# If one class has more black background/margin around the tissue than the
# other, the mean intensity gap can persist even after identical
# normalization -- and a model can key on "how much black border" as a
# trivial shortcut instead of anatomy.
def background_fraction(paths, max_n=300, threshold=15):
    fracs = []
    for p in paths[:max_n]:
        with PILImage.open(p) as img:
            arr = np.array(img.convert('L'), dtype=np.float32)
        fracs.append((arr < threshold).mean())  # fraction of near-black pixels
    return float(np.mean(fracs)), float(np.std(fracs))

for split_name, paths_list, labels_list in [
    ('train', train_paths, train_labels),
    ('val', val_paths, val_labels),
]:
    print(f'--- {split_name} ---')
    for label_id, label_name in enumerate(CLASS_NAMES):
        class_paths = [p for p, l in zip(paths_list, labels_list) if l == label_id]
        avg_frac, std_frac = background_fraction(class_paths)
        print(f'{label_name}: avg background fraction (avg, std) = ({avg_frac:.3f}, {std_frac:.3f})')
    print()

--- train ---
fake: avg background fraction (avg, std) = (0.414, 0.148)
real: avg background fraction (avg, std) = (0.389, 0.156)

--- val ---
fake: avg background fraction (avg, std) = (0.423, 0.150)
real: avg background fraction (avg, std) = (0.382, 0.142)



In [9]:
MODEL_NAME = 'google/vit-base-patch16-224'
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

def transform_batch(examples):
    encoded = processor(images=[image.convert('RGB') for image in examples['image']], return_tensors='pt')
    encoded['labels'] = examples['label']
    return encoded

train_ds = train_ds.with_transform(transform_batch)
val_ds = val_ds.with_transform(transform_batch)
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}
model = ViTForImageClassification.from_pretrained(MODEL_NAME, num_labels=2, label2id=label2id, id2label=id2label, ignore_mismatched_sizes=True)


C:\Users\aayus\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aayus\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1

In [16]:
import torch.nn as nn
from datetime import datetime
from transformers import Trainer, EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    return {'accuracy': accuracy_score(labels, predictions), 'precision': precision, 'recall': recall, 'f1': f1}

# Class weights, computed from whatever the actual training split turns out
# to be -- this is NOT hardcoded, so it self-adjusts as your data changes.
class_counts = Counter(train_labels)
n_classes = len(CLASS_NAMES)
total = sum(class_counts.values())
class_weights = [total / (n_classes * class_counts[i]) for i in range(n_classes)]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print('Class weights:', dict(zip(CLASS_NAMES, class_weights)))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Use a new Drive-backed output directory for every run. This avoids loading
# stale checkpoints created by another Transformers version or data split.
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = str(PROJECT_DIR / f'medical_deepfake_vitb16_model_{RUN_ID}')
HAS_GPU = torch.cuda.is_available()
BATCH_SIZE = 16 if HAS_GPU else 4
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3  # stop if val accuracy hasn't improved for 3 eval rounds
print('Training device:', 'GPU' if HAS_GPU else 'CPU', f'| batch size: {BATCH_SIZE} | epochs: {EPOCHS}')
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS, weight_decay=0.01, load_best_model_at_end=True,
    metric_for_best_model='accuracy', save_total_limit=2, report_to='none', remove_unused_columns=False
)
trainer = WeightedTrainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

Class weights: {'fake': 1.1588628762541806, 'real': 0.8794416243654822}
Training device: GPU | batch size: 16 | epochs: 10


In [17]:
# Never resume a checkpoint from an earlier run.
trainer.train(resume_from_checkpoint=False)
metrics = trainer.evaluate()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(metrics)
print(f'Model saved to {OUTPUT_DIR}')


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.000057,1.000000,1.000000,1.000000,1.000000
2,No log,0.000023,1.000000,1.000000,1.000000,1.000000
3,0.008552,0.000014,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Diagnostic: visualize what the trained model attends to

If the model is keying on borders, margins, or compression artifacts rather than tissue structure, this usually shows up as attention concentrated on image edges or a uniform background region instead of anatomy. Run this on a few real and a few fake validation images.


In [ ]:
import matplotlib.pyplot as plt

def show_attention(image_path, model_dir=OUTPUT_DIR, layer=-1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    local_model = ViTForImageClassification.from_pretrained(model_dir, output_attentions=True).to(device)
    local_model.eval()
    with Image.open(image_path) as image:
        rgb = image.convert('RGB')
        inputs = processor(images=rgb, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = local_model(**inputs)
    attn = outputs.attentions[layer][0]  # (heads, tokens, tokens)
    cls_attn = attn.mean(0)[0, 1:]  # average over heads; CLS token's attention to each patch
    grid_size = int(cls_attn.shape[0] ** 0.5)
    attn_map = cls_attn.reshape(grid_size, grid_size).detach().cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(rgb)
    axes[0].set_title('Input')
    axes[0].axis('off')
    axes[1].imshow(rgb)
    axes[1].imshow(attn_map, cmap='jet', alpha=0.5, extent=(0, rgb.width, rgb.height, 0))
    axes[1].set_title('CLS attention (last layer)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    pred_idx = int(outputs.logits.softmax(-1).argmax())
    print('Predicted:', local_model.config.id2label[pred_idx])

# Try a few examples of each class, e.g.:
# real_val = [p for p, l in zip(val_paths, val_labels) if l == label2id['real']]
# fake_val = [p for p, l in zip(val_paths, val_labels) if l == label2id['fake']]
# show_attention(real_val[0])
# show_attention(fake_val[0])
# If attention consistently pools on a corner, border, or flat background
# patch rather than brain tissue, that confirms a shortcut is being learned.


In [ ]:
def predict_image(image_path, model_dir=OUTPUT_DIR):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    local_model = ViTForImageClassification.from_pretrained(model_dir).to(device)
    with Image.open(image_path) as image:
        inputs = processor(images=image.convert('RGB'), return_tensors='pt').to(device)
    with torch.no_grad():
        probabilities = torch.softmax(local_model(**inputs).logits, dim=-1)[0].cpu().numpy()
    index = int(np.argmax(probabilities))
    return local_model.config.id2label[index], float(probabilities[index]), probabilities

# For a PNG/JPG image produced by prepare_dataset.py:
# predict_image('medical_deepfake_data/processed/val/fake/example.png')


In [ ]:
# Export the fine-tuned model and matching image processor where the backend expects them.
# Run this notebook with the repository root or backend folder as the working directory.
BACKEND_DIR = Path.cwd()
if BACKEND_DIR.name.lower() != "backend":
    BACKEND_DIR = BACKEND_DIR / "backend"
if not (BACKEND_DIR / "server.py").is_file():
    raise FileNotFoundError(
        f"Could not locate backend/server.py from {Path.cwd()}. "
        "Run this notebook with the repository root or backend folder as its working directory."
    )

BACKEND_MODEL_DIR = BACKEND_DIR / "models" / "medical_deepfake_vitb16"
BACKEND_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(BACKEND_MODEL_DIR))
processor.save_pretrained(str(BACKEND_MODEL_DIR))
print(f"Backend model exported to: {BACKEND_MODEL_DIR}")
